# BIOT 6900 · Module 5, Part 3 — Assignment

**Computational Lab 5 · 100 points · 3% of your course grade**

## What this assignment is

In Part 1 you built four variant predictors from scratch on a toy gene. In Part 2 we ran the real tools across five pharmacogenes together. Now you run the whole workflow yourself on **one gene**, and defend every choice in writing.

**Your gene.** Set `MY_GENE` below to the gene assigned to you on Canvas. If you were not assigned one, pick any of the five and say which in your README.

**What you hand in.** This notebook, one CSV it produces, and a 3 to 4 page report.

Maps to LO1 (apply industry-standard tools), LO3 (interpret results in biological context), LO4 (evaluate model performance, limitations and validation requirements) and LO5 (communicate through professional documentation).

## How it is graded

This follows the course 60 / 40 philosophy: **60 points for computational execution** and **40 points for interpretation and communication.**

| Component | Points | Type |
|---|---|---|
| **Q1 · Fetch and quality-check your gene** | **15** | execution |
| **Q2 · Audit tool coverage** | **10** | execution |
| **Q3 · Frequency filter with a justified threshold** | **15** | execution |
| **Q4 · Calibrate a predictor into evidence** | **15** | execution |
| **Q5 · Classify the uncertain variants and export** | **5** | execution |
| **Q6 · Report, four questions** | **40** | interpretation |
| **Total** | **100** | |

**Partial credit** is awarded for a correct conceptual approach even with minor technical errors. If a step would not run, say in your README what you were trying to do and what happened. Explaining your reasoning earns credit.

**Disclosed fallbacks cost you nothing.** If you could not get an AlphaGenome key, or your gene was too small to calibrate, say so and explain the consequence. Taking a fallback silently is what loses marks.

## Before you start

- You need the `biot6900` environment and an internet connection for the first run.
- Work through **Part 2 first**. This assignment reuses its functions and assumes you have read its commentary.
- An AlphaGenome key is optional. Section 7 of the Part 2 lab guide has the setup steps.

## Due date and late policy

**Due Thursday, October 22 at 6:00 pm.** Per the syllabus: −10% up to 1 day late, −20% up to 2 days, −50% up to 3 days, and no credit after the 3rd day unless we have arranged an alternative in advance.

---
## Step 0 · Setup

Run this cell. It is the same machinery you used in Part 2, collected in one place: the API fields, the parsing helpers, `fetch_gene()`, `numeric()`, `clinvar_label()` and `calibrate()`.

You do not need to change anything here except `MY_GENE`.

In [1]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

/opt/anaconda3/envs/biot6900/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [2]:
MY_GENE = "CYP2C19"        # <-- set this to YOUR assigned gene
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on CYP2C19


---
## Q1 · Fetch and quality-check your gene  (15 points)

Part 2 fetched five genes for you to read. Now do it for yours, and then **check what came back**. A truncated or empty fetch is the failure mode that silently ruins everything downstream, and it does not raise an error.

**Cache your result** to `data/<YOUR_GENE>.tsv` and load from there on later runs. Re-downloading on every `Restart & Run All` is slow and is poor practice against a free public API.

**Three quality checks you must run and report:**

1. **Count.** How many variants came back? A gene with 3 variants means your query string is wrong.
2. **Anchors.** Did the known star allele for your gene appear? `ANCHORS` lists them. If your gene has one and it is missing, your fetch is incomplete.
3. **Score columns.** Are CADD, REVEL and gnomAD populated at all, or did everything come back empty?

In [3]:
# Q1 · fetch (or load from cache), clean, label, quality-check

if MY_FILE.exists() and not FORCE_REFETCH:
    G = pd.read_csv(MY_FILE, sep="\t")
    print(f"loaded {MY_FILE} from cache")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)
    print(f"fetched and cached to {MY_FILE}")
G = numeric(G)
G["label"] = G.clinvar_sig.map(clinvar_label)

# Check 1 · count. A handful of rows means the query string is wrong;
# 1000 means the result cap truncated the gene.
print(f"\n[1] COUNT: {len(G)} variants for {MY_GENE}")
assert 20 <= len(G) < 1000, "suspicious count: check the query / result cap"
print("    ClinVar labels:", G.label.value_counts().to_dict())

# Check 2 · anchors. The star alleles that should be in any complete fetch.
mine = {rs: name for rs, name in ANCHORS.items() if name.startswith(MY_GENE)}
print(f"\n[2] ANCHORS for {MY_GENE}:")
for rs, name in mine.items():
    hit = G[G.rsid == rs]
    print(f"    {name:12s} {rs:11s} {'FOUND' if len(hit) else 'MISSING'}"
          + (f"  ({len(hit)} rows: {', '.join(hit.hgvs_g)})" if len(hit) else ""))
assert all((G.rsid == rs).any() for rs in mine), "anchor allele missing: fix the fetch"

# Check 3 · score columns. Populated at all, or silently empty?
print("\n[3] SCORE COLUMNS (non-null / total):")
for c in ["cadd_phred", "revel", "alphamissense", "gnomad_af"]:
    n = G[c].notna().sum()
    print(f"    {c:14s} {n:3d} / {len(G)}  ({n/len(G):5.1%})")
    assert n > 0, f"{c} came back entirely empty"

loaded data/CYP2C19.tsv from cache

[1] COUNT: 80 variants for CYP2C19
    ClinVar labels: {'other': 38, 'VUS': 33, 'B/LB': 9}

[2] ANCHORS for CYP2C19:
    CYP2C19*2    rs4244285   FOUND  (2 rows: chr10:g.94781859G>A, chr10:g.94781859G>T)
    CYP2C19*17   rs12248560  FOUND  (1 rows: chr10:g.94761900C>T)

[3] SCORE COLUMNS (non-null / total):
    cadd_phred      68 / 80  (85.0%)
    revel           54 / 80  (67.5%)
    alphamissense   54 / 80  (67.5%)
    gnomad_af       55 / 80  (68.8%)


---
## Q2 · Audit tool coverage  (10 points)

Work out what fraction of your gene's variants each tool could score, broken down by consequence class. You will be asked about the gaps in the report.

Use `TOOLS = ["cadd_phred", "revel", "alphamissense"]`, and add `"avi"` if you ran the Atlas in Q5.

In [4]:
# Q2 · which tool could score which consequence class?

TOOLS = ["cadd_phred", "revel", "alphamissense"]
if "avi" in G.columns and G["avi"].notna().any():
    TOOLS.append("avi")

cons = G.consequence.fillna("(unannotated: CNV / large deletion)")
cov = G[TOOLS].groupby(cons).apply(lambda s: s.notna().sum())
cov.insert(0, "n", cons.value_counts())
cov = cov.sort_values("n", ascending=False)
print(f"variants scored per tool, by consequence class ({MY_GENE})\n")
print(cov.to_string())

print("\noverall fraction scored")
for t in TOOLS:
    print(f"    {t:14s} {G[t].notna().mean():6.1%}   ({G[t].notna().sum()}/{len(G)})")

no_revel = cov.index[cov.revel == 0].tolist()
print(f"\nclasses with ZERO REVEL scores ({len(no_revel)}):")
for c in no_revel:
    print(f"    {c}  (n={cov.loc[c, 'n']})")
print(f"\n{(cov.revel == 0).mul(cov.n).sum()} variants sit in classes no missense predictor can see.")
print("Blank here means 'could not score', not 'scored benign'.")

variants scored per tool, by consequence class (CYP2C19)

                                      n  cadd_phred  revel  alphamissense
consequence                                                              
missense_variant                     54          53     53             54
(unannotated: CNV / large deletion)   8           0      0              0
synonymous_variant                    7           6      0              0
upstream_gene_variant                 5           4      0              0
intergenic_region                     1           1      0              0
intron_variant                        1           1      0              0
splice_donor_variant                  1           0      0              0
start_lost                            1           1      1              0
stop_gained                           1           1      0              0
stop_lost                             1           1      0              0

overall fraction scored
    cadd_phred      85.0%   (

---
## Q3 · Frequency filter, with a threshold you can defend  (15 points)

Choose a maximum credible allele frequency for your gene and **justify it from the biology of this gene and the drugs it affects**, not from habit.

**A warning specific to pharmacogenes.** Reduced-function pharmacogene alleles are frequently **common**. CYP2C19\*2 sits around 15% in many populations and VKORC1 −1639G>A near 40% in some. A rare-disease filter of 0.1% would discard the most clinically important variants in these genes.

Whatever threshold you pick, you must inspect what it throws away.

In [5]:
# Q3 · a frequency filter for a PHARMACOGENE, not a rare-disease gene
#
# CYP2C19 loss of function causes no disease. It only matters when a patient
# takes a CYP2C19 substrate (clopidogrel, voriconazole, PPIs, citalopram /
# sertraline), so there has been no purifying selection to keep functional
# alleles rare. *2 is ~15% in Europeans/Africans and ~30% in East Asians
# (higher in Oceania), *17 ~20%. A threshold must keep all of those.
# 0.5 removes only sites where the "alt" allele is the MAJORITY allele, i.e.
# where the reference genome carries the minor allele and the "variant" is the
# population norm. Those cannot be a rare functional explanation for anything.

MAX_AF = 0.5

def af_filter(df, max_af):
    """Keep variants with AF <= max_af. Missing AF is KEPT: absent from gnomAD
    is not evidence of being common."""
    keep = df.gnomad_af.isna() | (df.gnomad_af <= max_af)
    return df[keep], df[~keep]

show = ["rsid", "consequence", "aa_change", "cadd_phred", "gnomad_af", "clinvar_sig"]
is_functional = lambda d: d.clinvar_sig.fillna("").str.contains("drug response|athogenic", case=False) \
                          | d.rsid.isin(ANCHORS)

kept, removed = af_filter(G, MAX_AF)
print(f"MAX_AF = {MAX_AF}: removes {len(removed)} of {len(G)} variants "
      f"({G.gnomad_af.isna().sum()} with no gnomAD AF are kept)\n")
print(removed[show].to_string(index=False))
lost = removed[is_functional(removed)]
print(f"\nP/LP, drug-response or anchor variants removed: {len(lost)}")
if len(lost):
    print(lost[show].to_string(index=False))

# contrast: what a rare-disease 0.1% filter would have done
_, removed_rare = af_filter(G, 0.001)
lost_rare = removed_rare[is_functional(removed_rare)]
print(f"\n--- contrast: a 0.001 rare-disease filter would remove {len(removed_rare)} variants,"
      f" {len(lost_rare)} of them P/LP / drug response / anchors ---")
print(lost_rare.assign(anchor=lost_rare.rsid.map(ANCHORS).fillna(""))
      [show + ["anchor"]].sort_values("gnomad_af", ascending=False).to_string(index=False))

G = kept.reset_index(drop=True)
print(f"\nG now holds {len(G)} variants")

MAX_AF = 0.5: removes 2 of 80 variants (25 with no gnomAD AF are kept)

      rsid        consequence aa_change  cadd_phred  gnomad_af clinvar_sig
rs17885098 synonymous_variant       NaN         NaN   0.919465      Benign
 rs3758581   missense_variant     I331V         NaN   0.952250      Benign

P/LP, drug-response or anchor variants removed: 0

--- contrast: a 0.001 rare-disease filter would remove 19 variants, 14 of them P/LP / drug response / anchors ---
       rsid           consequence aa_change  cadd_phred  gnomad_af   clinvar_sig     anchor
 rs12248560 upstream_gene_variant       NaN       3.031   0.200120 Benign; other CYP2C19*17
 rs12777823     intergenic_region       NaN       0.725   0.190437 drug response           
 rs12769205        intron_variant       NaN       5.310   0.173336 drug response           
  rs4244285    synonymous_variant       NaN       5.686   0.167953 drug response  CYP2C19*2
  rs7902257 upstream_gene_variant       NaN         NaN   0.022133 drug respo

---
## Q4 · Calibrate a predictor into evidence  (15 points)

Pick the predictor with the best coverage of your gene and calibrate it with `calibrate()`, as in Part 2 Section 8. Report the likelihood ratio and the implied evidence strength per band.

Then **look hard at the denominators**. A likelihood ratio computed on four variants is not evidence; it is noise with a decimal point.

**If your gene has too few labelled variants to calibrate**, which is a real possibility for the smaller genes, do not force it. Fetch a second gene and pool, or state that calibration is not possible for your gene and explain why. Either is acceptable if you **disclose it**.

In [6]:
# Q4 · calibrate CADD into evidence
#
# Why CADD: it is the only tool that scores every consequence class in this
# gene (Q2), including the synonymous, start/stop and regulatory variants that
# REVEL and AlphaMissense cannot see.

SCORE_COL = "cadd_phred"
BANDS     = [0, 10, 20, 25, 30, 50]

labelled = G[G.label.isin(["P/LP", "B/LB"]) & G[SCORE_COL].notna()]
n_p, n_b = (labelled.label == "P/LP").sum(), (labelled.label == "B/LB").sum()
print(f"{MY_GENE} alone: {n_p} P/LP and {n_b} B/LB variants with a {SCORE_COL} score")

# CYP2C19 cannot be calibrated on its own. ClinVar files its functional alleles
# as "drug response" (label "other"), not P/LP, so there are no positives and
# every LR would be 0 or undefined. Route taken: POOL with the other four Module 5
# pharmacogenes, fetched and cached exactly like Q1, and frequency-filtered with
# the same MAX_AF.
POOL_WITH = [g for g in GENES if g != MY_GENE]
if n_p < 10 or n_b < 10:
    print(f"  -> too few labelled variants to calibrate. Pooling with {', '.join(POOL_WITH)}.\n")
    parts = [G.assign(src_gene=MY_GENE)]
    for g in POOL_WITH:
        f = DATA / f"{g}.tsv"
        d = pd.read_csv(f, sep="\t") if f.exists() else fetch_gene(g)
        if not f.exists():
            d.to_csv(f, sep="\t", index=False)
        d = numeric(d)
        d["label"] = d.clinvar_sig.map(clinvar_label)
        parts.append(af_filter(d, MAX_AF)[0].assign(src_gene=g))
    CAL_SET = pd.concat(parts, ignore_index=True)
else:
    CAL_SET = G.assign(src_gene=MY_GENE)

lab = CAL_SET[CAL_SET.label.isin(["P/LP", "B/LB"]) & CAL_SET[SCORE_COL].notna()]
print("labelled variants in the calibration set, by gene:")
print(pd.crosstab(lab.src_gene, lab.label, margins=True).to_string())

cal = calibrate(CAL_SET, SCORE_COL, BANDS)
print(f"\ncalibration of {SCORE_COL}: rests on {cal.attrs['n_p']} P/LP + {cal.attrs['n_b']} B/LB "
      f"= {cal.attrs['n_p'] + cal.attrs['n_b']} labelled variants\n")

# Sensitivity check: an LR of 0 or inf comes from an empty cell, not from a
# measured effect. Add 0.5 to every cell (Haldane) to see a finite estimate.
cal["LR+_haldane"] = (((cal.n_PLP + .5) / (cal.attrs["n_p"] + .5))
                      / ((cal.n_BLB + .5) / (cal.attrs["n_b"] + .5))).round(2)
cal["small_n"] = np.where(cal.n < 10, "<10  !", "")
print(cal.to_string())

small = cal[cal.n < 10]
print(f"\nbands resting on fewer than 10 variants: {len(small)}"
      + (f"  -> {', '.join(map(str, small.index))}" if len(small) else ""))
print(f"bands with an empty cell (LR = 0 or inf): {int(((cal.n_PLP == 0) | (cal.n_BLB == 0)).sum())}")
print(f"\nCAVEAT: {lab[lab.label == 'P/LP'].src_gene.value_counts().get('DPYD', 0)} of "
      f"{cal.attrs['n_p']} positives are DPYD. This is mostly a DPYD calibration "
      f"transferred to {MY_GENE}, which contributes {(lab.src_gene == MY_GENE).sum()} labelled variants.")

CYP2C19 alone: 0 P/LP and 5 B/LB variants with a cadd_phred score
  -> too few labelled variants to calibrate. Pooling with CYP2C9, VKORC1, DPYD, TPMT.

labelled variants in the calibration set, by gene:
label     B/LB  P/LP  All
src_gene                 
CYP2C19      5     0    5
CYP2C9       7     1    8
DPYD        91    39  130
TPMT         4     0    4
VKORC1      11     9   20
All        118    49  167

calibration of cadd_phred: rests on 49 P/LP + 118 B/LB = 167 labelled variants

           n  n_PLP  n_BLB   LR+                evidence  LR+_haldane small_n
band                                                                         
(0, 10]   76      2     74  0.07        moderate, benign         0.08        
(10, 20]  34      1     33  0.07        moderate, benign         0.11        
(20, 25]  17      8      9  2.14  supporting, pathogenic         2.14        
(25, 30]   6      4      2  4.82    moderate, pathogenic         4.31  <10  !
(30, 50]  34     34      0   inf      s

---
## Q5 · Classify the uncertain variants and export  (5 points)

Take the VUS rows in your gene, apply your calibrated evidence, and export the result.

**This is the entire point of calibration.** A VUS whose score falls in a band with an LR near 1 **stays a VUS**, because the predictor genuinely told you nothing. Only a large or small LR should move anything.

In [7]:
# Q5 · apply the calibrated evidence to the VUS, and export
#
# Rule: a VUS takes the evidence of its CADD band ONLY if that band rests on at
# least 10 labelled variants. A band with fewer is reported but not applied,
# because its LR is noise. "supporting" evidence alone does not reclassify
# under ACMG, so the call is phrased as a lean, never as P/LP or B/LB.

MIN_N = 10
vus = G[G.label == "VUS"].copy()
vus["band"] = pd.cut(vus[SCORE_COL], BANDS)
band_ev = cal.evidence.where(cal.n >= MIN_N, "insufficient data (band n<10)")
vus["band_LR"] = vus.band.map(cal["LR+"]).astype(float)
vus["evidence"] = vus.band.map(band_ev).astype(object)
vus.loc[vus[SCORE_COL].isna(), "evidence"] = "not scored"

def call(ev):
    if "pathogenic" in ev: return "VUS, leaning pathogenic"
    if "benign" in ev:     return "VUS, leaning benign"
    return "VUS (unchanged)"
vus["provisional_call"] = vus.evidence.map(call)
vus["direction"] = vus.provisional_call.map(
    {"VUS, leaning pathogenic": "towards pathogenic",
     "VUS, leaning benign": "towards benign"}).fillna("stays put")

print(f"{len(vus)} VUS in {MY_GENE}\n")
print(vus.groupby(["band", "evidence"], observed=True, dropna=False).size().to_string())
print(f"  (no {SCORE_COL} score: {vus[SCORE_COL].isna().sum()})\n")
print(vus.direction.value_counts().reindex(
    ["towards pathogenic", "towards benign", "stays put"], fill_value=0).to_string())

OUT = pathlib.Path(f"pgx_classified_{MY_GENE}.csv")
cols = ["rsid", "hgvs_g", "gene", "consequence", "aa_change", SCORE_COL, "revel",
        "alphamissense", "gnomad_af", "band_LR", "evidence", "provisional_call"]
vus[cols].sort_values(SCORE_COL, ascending=False).to_csv(OUT, index=False)
print(f"\nwrote {len(vus)} rows to {OUT}")

33 VUS in CYP2C19

band          evidence                     
(0.0, 10.0]   moderate, benign                 8
(10.0, 20.0]  moderate, benign                 8
(20.0, 25.0]  supporting, pathogenic           8
(25.0, 30.0]  insufficient data (band n<10)    3
NaN           not scored                       6
  (no cadd_phred score: 6)

direction
towards pathogenic     8
towards benign        16
stays put              9

wrote 33 rows to pgx_classified_CYP2C19.csv


---
## Q6 · The Atlas and the disagreements  (code given; your writing is graded in the report)

Run both cells. The first queries the AlphaGenome Atlas for your gene; the second applies the discordance rules from Part 2 Section 10.

**If you do not have an AlphaGenome key**, set `SKIP_ATLAS = True`. The second cell still works and you answer the Atlas part of the report from reasoning. No marks are lost for a disclosed skip.

In [8]:
SKIP_ATLAS   = True     # no AlphaGenome key available: disclosed in README/report
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"rsid": r.rsid,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="rsid", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")

SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.

non-missense variants in CYP2C19, which is what the Atlas is for: 25
consequence
synonymous_variant       6
upstream_gene_variant    5
start_lost               1
stop_gained              1
intron_variant           1
splice_donor_variant     1
stop_lost                1
intergenic_region        1

[Atlas status: skipped]


In [9]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

discordant variants in CYP2C19

flag_cadd_vs_revel      3
flag_revel_vs_am        6
flag_scored_but_vus     0
flag_common_but_plp     4
flag_no_tool_at_all    10

21 variants trip at least one rule

       rsid           consequence                         consequence_all aa_change  cadd_phred  revel  alphamissense  avi  gnomad_af label
        NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  NaN        NaN   VUS
  rs7902257 upstream_gene_variant intergenic_region;upstream_gene_variant       NaN         NaN    NaN            NaN  NaN   0.022133 other
        NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  NaN        NaN  B/LB
 rs41291556      missense_variant                        missense_variant     W120R      25.700  0.440         0.9402  NaN   0.001571 other
rs145328984      missense_variant                        missense_variant      R73C      26.000  0.12

---
## Q6 · Your report  (40 points)

Write **3 to 4 pages** answering the four questions below. Marks are for reasoning, not for reaching a particular conclusion. A confident, clearly argued wrong answer scores better than a hedged list of possibilities.

### 6.1 · Thresholds and calibration  (10 points)

Which allele-frequency threshold did you choose in Q3, and why is it right for *this gene* rather than for a rare-disease gene? What did your 0.001 contrast discard? Which predictor did you calibrate and why that one? Report the likelihood ratios, state how many labelled variants they rest on, and say what that does to your confidence in them.

### 6.2 · Coverage gaps and the regulatory genome  (10 points)

Which consequence classes in your gene could no missense predictor score? Pick **one specific unscored variant**, name it, and say what evidence you would need to classify it and which tool or assay would produce that evidence.

Then address the Atlas. If you ran it: what did it add, and what did the `top_feature` attributions say was driving the top scores? Remember that a score driven by `CACTUS_241_WAY` means conservation is carrying it and the mechanism is unresolved, which is weaker than one driven by `MAX_ABS_DNASE`. If you did not run it: say what it could in principle have told you about which specific variants in your gene, and why nothing else in the course can.

### 6.3 · A discordant variant, read mechanistically  (12 points)

Choose **one** variant from the Q6 discordance output. State which rule it tripped, what each tool said about it, and what biology could produce that pattern. Argue for one explanation over the alternatives.

The question to answer is not *which tool is right*. It is *what would have to be true about this variant for both outputs to make sense*.

### 6.4 · What you could not determine  (8 points)

Name **two** things your analysis cannot establish about your most interesting variant, drawing on the lecture's list: penetrance, mechanism and direction, zygosity and phase, tissue and timing, modifiers including phenoconversion, and clinical utility.

Then name the **single experiment or dataset** that would resolve your most interesting call, and say what result would change your mind.

---
## Report rubric

Each of the four questions is marked on the same four-level scale, scaled to its point value.

| Level | What it looks like |
|---|---|
| **Excellent** (90-100% of the points) | Answers the question directly with specifics from *their own* output. Names variants, quotes their own numbers. Commits to an interpretation and defends it. Acknowledges the limits of the evidence without hiding behind them. |
| **Good** (75-89%) | Correct and specific, but stops short of an argument. Describes what the numbers are rather than what they mean. Limitations listed rather than weighed. |
| **Developing** (55-74%) | Generic. Could have been written without running the notebook. Repeats lecture phrasing rather than engaging with this gene. Numbers quoted without interpretation, or interpretation offered without numbers. |
| **Insufficient** (<55%) | Does not answer the question asked, contradicts their own output, or treats a predictor score as a verdict about a patient. |

**Across all four questions, these specifically earn credit:**

- Quoting your own numbers rather than the ones in the lab guide
- Naming a specific variant rather than describing a category
- Disclosing a fallback and explaining its cost
- Distinguishing *could not score* from *scored and found nothing*

**And these specifically lose it:**

- Treating a blank score as a benign result
- Reporting a likelihood ratio from a tiny band without caveat
- Calling a variant pathogenic on the strength of predictor scores alone
- Listing possible explanations instead of arguing for one

---
## Submit

1. `Kernel → Restart & Run All` and confirm the notebook runs cleanly top to bottom.
2. Commit to your `biot6900` repository:
   - this notebook
   - `pgx_classified_<YOUR_GENE>.csv`
   - your report, as PDF
   - a short `README.md` with your name, your gene, whether you ran the Atlas, and anything that did not work
3. Push, confirm the files appear on github.com, then **post your repository link on Canvas**.

**Do not commit your AlphaGenome API key.** It belongs in your environment, never in this notebook. A key in a commit stays in the history even after you delete the line.

**Due Thursday, October 22 at 6:00 pm.** Late policy per syllabus.

*BIOT 6900 · Module 5, Part 3 · Fall 2026*